In [3]:
import os
import json
import glob
import pandas as pd
import xlrd
import re
from pathlib import Path
import numpy as np
import xlsxwriter
import openpyxl
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter
from openpyxl.styles import Alignment, Border, Side, PatternFill

In [2]:
def correct_distt_name(incorrect_distt):
    with open("distt_correction.json", "r") as file:
        distt_correction = json.load(file)

    for correct, incorrect_list in distt_correction.items():
        for incorrect in incorrect_list:
            if incorrect in incorrect_distt:
                correct_distt = correct
                return correct_distt

    return incorrect_distt

In [3]:
def correct_crop_name(incorrect_crop):
    with open("crop_correction.json", "r") as file:
        crop_correction = json.load(file)

    for correct, incorrect_list in crop_correction.items():
        for incorrect in incorrect_list:
            if incorrect.lower() in incorrect_crop.lower():
                if '(UI)' in incorrect_crop:
                    crop = correct + '_UI'
                elif '(I)' in incorrect_crop or '(IR)' in incorrect_crop:
                    crop = correct + '_IR'
                else:
                    crop = correct
                return crop
    
    return incorrect_crop

In [4]:
def get_season(crop):
    with open("crop_seq.json", "r") as file:
        crop_seq = json.load(file)

    for season, crop_dict in crop_seq.items():
        for cr in crop_dict.keys():
            if crop.lower() in cr.lower():
                return season
    return None

In [5]:
def get_first_row(worksheet, row_text_list):
    for text in row_text_list:
        text = text.lower()
        for row in worksheet.iter_rows():
            for cell in row:
                if cell.value and text in str(cell.value).lower():
                    return cell.row
    return -1

In [6]:
def get_first_col(worksheet, col_text_list):
    for text in col_text_list:
        text = text.lower()
        for row in worksheet.iter_rows():
            for cell in row:
                if cell.value and text in str(cell.value).lower():
                    return cell.column
    return -1

In [7]:
def get_SL_to_ML(directory, state):
    records = []

    #df to use
    df_state = pd.read_excel("ML_Template.xlsx", sheet_name="State")
    df_seasons = pd.read_excel("ML_Template.xlsx", sheet_name="Seasons")
    df_samples = pd.read_excel("ML_Template.xlsx", sheet_name="Samples")
    df_districts = pd.read_excel("ML_Template.xlsx", sheet_name="Districts")
    df_crops = pd.read_excel("ML_Template.xlsx", sheet_name="Crops")

    #df to create
    df_Vill = pd.DataFrame(columns=['STATENAME','SEASONNAME','SAMPLENAME','DISTRICTNAME',
                                    'CROPNAME','TALUKA','CIRCLE','VILLAGE'])
    
    df_ML = pd.DataFrame(columns=['YEAR','SEASONCODE','SEASONNAME','HSEASONNAME',
                                  'SAMPLE','SAMPLENAME','HSAMPLENAME','STATE',
                                  'STATENAME','SHORTSTATE','HSTATENAME','ROCODE',
                                  'RONAME','HRONAME','SROCODE','SRONAME','HSRONAME',
                                  'DISTRICT','DISTRICTNAME','HDISTRICTNAME',
                                  'SELORDER','EXPT','CROPCODE','CROPNAME',
                                  'HCROPNAME','STATUS','EXPTID',])
    
    # Get all .xlsx files in the directory
    excel_files = glob.glob(os.path.join(directory, "*.xlsx"))

    #for each excel workbook
    for file_path in excel_files:
        file_name = os.path.basename(file_path)
        incorrect_distt = Path(file_name).stem.title()

        distt = correct_distt_name(incorrect_distt)     ############################## usable
        # Determine file type and use appropriate library
        if file_path.endswith('.xlsx'):
            try:
                wb = load_workbook(file_path, data_only=True)
                #for each sheet in excel workbook
                for sheet_name in wb.sheetnames:
                    if "cent" in sheet_name.lower() or "sta" in sheet_name.lower():
                        st = 1 if "cent" in sheet_name.lower() else 2     ############################## usable
                        sample = df_samples[df_samples['SAMPLE']==st]['SAMPLENAME'].iloc[0]
                        
                        ws = wb[sheet_name]
                        
                        # setting which cells to scan
                        row_text_list = ['taluka', 'circle', 'vill', 'exp', 'os']
                        start_row = 0
                        title_row = 0
                        while start_row <= 0:
                            title_row = get_first_row(ws, row_text_list)
                            crop_row = title_row-1
                            start_row = title_row + 1
                            
                        col_text_list = ['vill']
                        start_col = 0
                        while start_col <= 0:
                            start_col = get_first_col(ws, col_text_list) + 1
                            taluka_col = start_col - 3
                            circle_col = start_col - 2
                            village_col = start_col - 1
                        
                        end_row = ws.max_row + 1
                        end_col = ws.max_column + 1

                        if start_row > 0 and start_col > 0:
                            for col_idx in range(start_col, end_col):
                                # get crop name and type
                                
                                incorrect_crop = ws.cell(row=crop_row, column=col_idx).value     ############################## usable
                                if isinstance(incorrect_crop, str) and incorrect_crop is not None:
                                    incorrect_crop = re.sub(r'\s+', '', incorrect_crop)
                                else:
                                    incorrect_crop = ws.cell(row=crop_row, column=col_idx-1).value
                                    if isinstance(incorrect_crop, str) and incorrect_crop is not None:
                                        incorrect_crop = re.sub(r'\s+', '', incorrect_crop)
                                    else:
                                        incorrect_crop = None
    
                                # set season
                                if incorrect_crop is not None:
                                    crop = correct_crop_name(incorrect_crop)
                                    crop_new = crop
                                    season = get_season(crop)
                                    
                                    
                                    plan = 0
                                    for row_idx in range(start_row, end_row):
                                        cell_val = ws.cell(row=row_idx, column=col_idx).value
                                        if isinstance(cell_val, str) and cell_val is not None:
                                            cell_val = cell_val.strip()
                                            
                                        cell_header = ws.cell(row=title_row, column=col_idx).value
                                        if isinstance(cell_header, str) and cell_header is not None:
                                            cell_header = cell_header.strip()
                                        
                                        if cell_val is not None:
                                            if taluka_col>0 and circle_col>0 and village_col>0:
                                                taluka = ws.cell(row=row_idx, column=taluka_col).value     ############################## usable
                                                circle = ws.cell(row=row_idx, column=circle_col).value     ############################## usable
                                                village = ws.cell(row=row_idx, column=village_col).value     ############################## usable

                                                if isinstance(taluka, str) and taluka is not None:
                                                    if isinstance(village, str) and village is not None:
                                                        #if taluka is not None and village is not None:
                                                        if 'exp' in str(cell_header).lower():
                                                            plan += cell_val     ############################## usable
                                                        if 'os' in str(cell_header).lower():
                                                            if 'A' in str(cell_val):
                                                                crop_new = crop + '_A'
                                                            elif 'B' in str(cell_val):
                                                                crop_new = crop + '_B'
        
                                                            m_digit  = re.search(r"\d+", str(cell_val))
                                                            if m_digit is not None:
                                                                selorder = m_digit.group()
        
                                                                if len(selorder) == 1:
                                                                    selorder = '0' + selorder
        
        
                                                                if state is not None:
                                                                    state = state.upper()
                                                                if season is not None:
                                                                    season = season.upper()
                                                                if sample is not None:
                                                                    sample = sample.upper()
                                                                if distt is not None:
                                                                    distt = distt.upper()
                                                                if crop_new is not None:
                                                                    crop_new = crop_new.upper()
                                                                if str(taluka) is not None:
                                                                    taluka = taluka.upper()
                                                                if str(circle) is not None:
                                                                    circle = circle.upper()
                                                                if str(village) is not None:
                                                                    village = village.upper()
                                                                
                                                                df_village_row = {
                                                                    'STATENAME': state,
                                                                    'SEASONNAME': season,
                                                                    'SAMPLENAME': sample,
                                                                    'DISTRICTNAME': distt,
                                                                    'CROPNAME': crop_new,
                                                                    'TALUKA': taluka,
                                                                    'CIRCLE': circle,
                                                                    'VILLAGE': village
                                                                }
                                                                df_Vill = pd.concat([df_Vill, pd.DataFrame([df_village_row])], 
                                                                                    ignore_index=True)
                                                                
                                                                df_ML_row = {
                                                                    'STATENAME': [state, state],
                                                                    'SEASONNAME': [season, season],
                                                                    'SAMPLENAME': [sample, sample],
                                                                    'DISTRICTNAME': [distt, distt],
                                                                    'CROPNAME': [crop_new, crop_new],
                                                                    'SELORDER': [selorder, selorder],
                                                                    'EXPT': ['1','2']
                                                                }
                                                                
                                                                df_ML = pd.concat([df_ML, pd.DataFrame(df_ML_row, dtype=str)], ignore_index=True)

            except Exception as e:
                print(f"Error reading {file_name}, {season}, {sample}, {distt}, {crop_new}, {village} (openpyxl): {e}")
                raise e
    df_ML['conflicting_col'] = (df_ML['key']
        .map(df_state.set_index('key')['conflicting_col'])
        .fillna(df_ML['conflicting_col'])
    )
    df_ML = pd.merge(df_ML,df_state,how='left',left_on='STATENAME',right_on='STATENAME')
    df_ML = pd.merge(df_ML,df_state,how='left',on='STATENAME')
    df_ML = pd.merge(df_ML,df_seasons,how='left',on='SEASONNAME')
    df_ML = pd.merge(df_ML,df_samples,how='left',on='SAMPLENAME')
    df_ML = pd.merge(df_ML,df_districts,how='left',on='DISTRICTNAME')
    df_ML = pd.merge(df_ML,df_crops,how='left',on='CROPNAME')
    
    return df_Vill, df_ML

In [8]:
if __name__ == "__main__":
    state = 'GUJARAT'
    curr_dir = Path.cwd()
    directory = curr_dir / "SL 2.0"
    excel_path = curr_dir / "SL_to_ML.xlsx"

    df_Vill, df_ML = get_SL_to_ML(directory, state)

    with pd.ExcelWriter(excel_path, engine='xlsxwriter') as writer:
        df_Vill.to_excel(writer, sheet_name="Villages", index=False)
        df_ML.to_excel(writer, sheet_name="ML", index=False)

In [70]:
df_state = pd.read_excel("ML_Template.xlsx", sheet_name="State", dtype=str)
df_seasons = pd.read_excel("ML_Template.xlsx", sheet_name="Seasons", dtype=str)
df_samples = pd.read_excel("ML_Template.xlsx", sheet_name="Samples", dtype=str)
df_districts = pd.read_excel("ML_Template.xlsx", sheet_name="Districts", dtype=str)
df_crops = pd.read_excel("ML_Template.xlsx", sheet_name="Crops", dtype=str)

In [71]:
df_ML = pd.DataFrame(columns=['YEAR','SEASONCODE','SEASONNAME','HSEASONNAME',
                              'SAMPLE','SAMPLENAME','HSAMPLENAME','STATE',
                              'STATENAME','SHORTSTATE','HSTATENAME','ROCODE',
                              'RONAME','HRONAME','SROCODE','SRONAME','HSRONAME',
                              'DISTRICT','DISTRICTNAME','HDISTRICTNAME',
                              'SELORDER','EXPT','CROPCODE','CROPNAME',
                              'HCROPNAME','STATUS','EXPTID',])

In [72]:
state = 'GUJARAT'
season = 'KHARIF'
sample = 'CENTRAL'
distt = 'PATAN'
crop_new = 'BAJRA'
selorder = '01'

In [73]:
df_ML_row = {
    'STATENAME': [state, state],
    'SEASONNAME': [season, season],
    'SAMPLENAME': [sample, sample],
    'DISTRICTNAME': [distt, distt],
    'CROPNAME': [crop_new, crop_new],
    'SELORDER': [selorder, selorder],
    'EXPT': ['1','2']
}

In [74]:
df_ML = pd.concat([df_ML, pd.DataFrame(df_ML_row)], ignore_index=True)

In [75]:
df_ML

,YEAR,SEASONCODE,SEASONNAME,HSEASONNAME,SAMPLE,SAMPLENAME,HSAMPLENAME,STATE,STATENAME,SHORTSTATE,...,DISTRICT,DISTRICTNAME,HDISTRICTNAME,SELORDER,EXPT,CROPCODE,CROPNAME,HCROPNAME,STATUS,EXPTID
0,NaN,NaN,KHARIF,NaN,NaN,CENTRAL,NaN,NaN,GUJARAT,NaN,...,NaN,PATAN,NaN,01,1,NaN,BAJRA,NaN,NaN,NaN
1,NaN,NaN,KHARIF,NaN,NaN,CENTRAL,NaN,NaN,GUJARAT,NaN,...,NaN,PATAN,NaN,01,2,NaN,BAJRA,NaN,NaN,NaN


In [76]:
df_ML['STATE'] = (df_ML['STATENAME']
        .map(df_state.set_index('STATENAME')['STATE'])
        .fillna(df_ML['STATE'])
    )
df_ML['SHORTSTATE'] = (df_ML['STATENAME']
        .map(df_state.set_index('STATENAME')['SHORTSTATE'])
        .fillna(df_ML['SHORTSTATE'])
    )
df_ML['HSTATENAME'] = (df_ML['STATENAME']
        .map(df_state.set_index('STATENAME')['HSTATENAME'])
        .fillna(df_ML['HSTATENAME'])
    )

In [77]:
df_ML['SAMPLE'] = (df_ML['SAMPLENAME']
        .map(df_samples.set_index('SAMPLENAME')['SAMPLE'])
        .fillna(df_ML['SAMPLE'])
    )
df_ML['HSAMPLENAME'] = (df_ML['SAMPLENAME']
        .map(df_samples.set_index('SAMPLENAME')['HSAMPLENAME'])
        .fillna(df_ML['HSAMPLENAME'])
    )

In [78]:
df_ML['DISTRICT'] = (df_ML['DISTRICTNAME']
        .map(df_districts.set_index('DISTRICTNAME')['DISTRICT'])
        .fillna(df_ML['DISTRICT'])
    )
df_ML['HDISTRICTNAME'] = (df_ML['DISTRICTNAME']
        .map(df_districts.set_index('DISTRICTNAME')['HDISTRICTNAME'])
        .fillna(df_ML['HDISTRICTNAME'])
    )
df_ML['ROCODE'] = (df_ML['DISTRICTNAME']
        .map(df_districts.set_index('DISTRICTNAME')['ROCODE'])
        .fillna(df_ML['ROCODE'])
    )
df_ML['RONAME'] = (df_ML['DISTRICTNAME']
        .map(df_districts.set_index('DISTRICTNAME')['RONAME'])
        .fillna(df_ML['RONAME'])
    )
df_ML['HRONAME'] = (df_ML['DISTRICTNAME']
        .map(df_districts.set_index('DISTRICTNAME')['HRONAME'])
        .fillna(df_ML['HRONAME'])
    )
df_ML['SROCODE'] = (df_ML['DISTRICTNAME']
        .map(df_districts.set_index('DISTRICTNAME')['SROCODE'])
        .fillna(df_ML['SROCODE'])
    )
df_ML['SRONAME'] = (df_ML['DISTRICTNAME']
        .map(df_districts.set_index('DISTRICTNAME')['SRONAME'])
        .fillna(df_ML['SRONAME'])
    )
df_ML['HSRONAME'] = (df_ML['DISTRICTNAME']
        .map(df_districts.set_index('DISTRICTNAME')['HSRONAME'])
        .fillna(df_ML['HSRONAME'])
    )

In [79]:
df_ML['CROPCODE'] = (df_ML['SEASONNAME_'+'CROPNAME']
        .map(df_crops.set_index('SEASONNAME_'+'CROPNAME')['CROPCODE'])
        .fillna(df_ML['CROPCODE'])
    )

KeyError: 'SEASONNAME_CROPNAME'

In [80]:
df_ML.iloc[0]

YEAR                   NaN
SEASONCODE             NaN
SEASONNAME          KHARIF
HSEASONNAME            NaN
SAMPLE                   1
SAMPLENAME         CENTRAL
HSAMPLENAME      केन्द्रीय
STATE                   24
STATENAME          GUJARAT
SHORTSTATE              GJ
HSTATENAME          गुजरात
ROCODE                 242
RONAME            VADODARA
HRONAME             वड़ोदरा
SROCODE               2421
SRONAME            MEHSANA
HSRONAME           महेसाणा
DISTRICT                03
DISTRICTNAME         PATAN
HDISTRICTNAME         पाटन
SELORDER                01
EXPT                     1
CROPCODE               NaN
CROPNAME             BAJRA
HCROPNAME              NaN
STATUS                 NaN
EXPTID                 NaN
Name: 0, dtype: object